# Modelado supervisado del Titanic
Comparamos Logistic Regression y Random Forest usando el dataset limpio generado a partir del EDA. El pipeline incluye imputacion, escalado y codificacion para que pueda reutilizarse desde el frontend.

In [ ]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

root = Path.cwd().parent if Path.cwd().name == 'scratch' else Path.cwd()
df = pd.read_csv(root / 'data' / 'titanic_clean.csv')
X = df.drop(columns='Survived')
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
numeric = X.select_dtypes(include='number').columns.tolist()
categorical = X.select_dtypes(exclude='number').columns.tolist()
preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric),
    ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))]), categorical),
])
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=2, random_state=42, n_jobs=-1),
}
results = {}
best_name, best_score, best_pipeline = None, -1, None
for name, estimator in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('classifier', estimator)])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    results[name] = {'accuracy': accuracy_score(y_test, predictions), 'f1': f1_score(y_test, predictions)}
    if results[name]['f1'] > best_score:
        best_name, best_score, best_pipeline = name, results[name]['f1'], pipeline
pd.DataFrame(results).T

In [ ]:
model_path = root / 'models' / 'titanic_model.joblib'
model_path.parent.mkdir(exist_ok=True)
joblib.dump(best_pipeline, model_path)
print(f'Mejor modelo: {best_name}')
print(f'Modelo guardado en: {model_path}')

## Uso local
Ejecuta `python train_model.py` para reentrenar y `python -m flask --app app run` para abrir la interfaz en `http://127.0.0.1:5000`.